In [1]:
import pandas as pd
import numpy as np
import joblib
from scipy.sparse import load_npz
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
movies = pd.read_csv("../data/processed/movies_clean.csv")

tfidf_matrix = joblib.load("../models/tfidf_matrix.pkl")
movie_indices = joblib.load("../models/movie_indices.pkl")

user_movie_sparse = load_npz("../models/user_movie_sparse.npz")
movie_to_index = joblib.load("../models/movie_to_index.pkl")
index_to_movie = joblib.load("../models/index_to_movie.pkl")

print("Movies:", movies.shape)
print("TF-IDF:", tfidf_matrix.shape)
print("Collaborative matrix:", user_movie_sparse.shape)

Movies: (58098, 3)
TF-IDF: (58098, 23)
Collaborative matrix: (283172, 18366)


In [3]:
movie_id = 1

if movie_id in movie_to_index:
    movie_index = movie_to_index[movie_id]

    movie_vector = user_movie_sparse[:, movie_index].T

    collaborative_scores = cosine_similarity(
        movie_vector,
        user_movie_sparse.T
    ).flatten()

    print("Similarity calculated successfully!")
    print("Number of scores:", len(collaborative_scores))
else:
    print("Movie ID not available")

Similarity calculated successfully!
Number of scores: 18366


In [4]:
user_movie_sparse = load_npz("../models/user_movie_sparse.npz")
movie_to_index = joblib.load("../models/movie_to_index.pkl")
index_to_movie = joblib.load("../models/index_to_movie.pkl")

In [5]:
def hybrid_recommend(movie_title, num_recommendations=10,
                     content_weight=0.6,
                     collaborative_weight=0.4):

    # Check movie
    if movie_title not in movie_indices:
        return f"Movie '{movie_title}' not found."

    # Get movie index
    movie_index = movie_indices[movie_title]

    # -----------------------------
    # 1. Content-based similarity
    # -----------------------------
    content_scores = cosine_similarity(
        tfidf_matrix[movie_index],
        tfidf_matrix
    ).flatten()

    # -----------------------------
    # 2. Collaborative similarity
    # -----------------------------
    movie_id = movies.iloc[movie_index]["movieId"]

    collaborative_score_map = {}

    if movie_id in movie_to_index:

        collaborative_index = movie_to_index[movie_id]

        collaborative_scores = cosine_similarity(
            user_movie_sparse[:, collaborative_index].T,
            user_movie_sparse.T
        ).flatten()

        for i, score in enumerate(collaborative_scores):
            recommended_movie_id = index_to_movie[i]
            collaborative_score_map[recommended_movie_id] = score

    # -----------------------------
    # 3. Create result dataframe
    # -----------------------------
    results = movies[[
        "movieId",
        "title",
        "genres"
    ]].copy()

    # Content score
    results["content_score"] = results.index.map(
        lambda x: content_scores[x]
    )

    # Collaborative score
    results["collaborative_score"] = results["movieId"].map(
        collaborative_score_map
    ).fillna(0)

    # -----------------------------
    # 4. Hybrid score
    # -----------------------------
    results["hybrid_score"] = (
        content_weight * results["content_score"]
        +
        collaborative_weight * results["collaborative_score"]
    )

    # Remove selected movie
    results = results[
        results["title"] != movie_title
    ]

    # Sort by hybrid score
    results = results.sort_values(
        "hybrid_score",
        ascending=False
    )

    # Return top recommendations
    return results.head(num_recommendations)

In [6]:
recommendations = hybrid_recommend(
    "Toy Story (1995)",
    10
)

recommendations

,movieId,title,genres,content_score,collaborative_score,hybrid_score
3028,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,1.000000,0.524543,0.809817
4791,4886,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,1.000000,0.469571,0.787828
4212,4306,Shrek (2001),Adventure|Animation|Children|Comedy|Fantasy|Ro...,0.937954,0.473106,0.752015
2210,2294,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy,1.000000,0.304370,0.721748
6272,6377,Finding Nemo (2003),Adventure|Animation|Children|Comedy,0.869137,0.460059,0.705506
3923,4016,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.000000,0.260124,0.704050
2271,2355,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy,0.869137,0.436958,0.696265
32293,134853,Inside Out (2015),Adventure|Animation|Children|Comedy|Drama|Fantasy,0.975194,0.232506,0.678119
11899,53121,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy,1.000000,0.183886,0.673554
5122,5218,Ice Age (2002),Adventure|Animation|Children|Comedy,0.869137,0.340191,0.657559


In [8]:
recommendations[
    [
        "title",
        "genres",
        "content_score",
        "collaborative_score",
        "hybrid_score"
    ]
]

,title,genres,content_score,collaborative_score,hybrid_score
3028,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,1.000000,0.524543,0.809817
4791,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,1.000000,0.469571,0.787828
4212,Shrek (2001),Adventure|Animation|Children|Comedy|Fantasy|Ro...,0.937954,0.473106,0.752015
2210,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy,1.000000,0.304370,0.721748
6272,Finding Nemo (2003),Adventure|Animation|Children|Comedy,0.869137,0.460059,0.705506
3923,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.000000,0.260124,0.704050
2271,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy,0.869137,0.436958,0.696265
32293,Inside Out (2015),Adventure|Animation|Children|Comedy|Drama|Fantasy,0.975194,0.232506,0.678119
11899,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy,1.000000,0.183886,0.673554
5122,Ice Age (2002),Adventure|Animation|Children|Comedy,0.869137,0.340191,0.657559


In [9]:
recommendations["content_score"] = recommendations["content_score"].round(3)
recommendations["collaborative_score"] = recommendations["collaborative_score"].round(3)
recommendations["hybrid_score"] = recommendations["hybrid_score"].round(3)

recommendations

,movieId,title,genres,content_score,collaborative_score,hybrid_score
3028,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,1.000,0.525,0.810
4791,4886,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,1.000,0.470,0.788
4212,4306,Shrek (2001),Adventure|Animation|Children|Comedy|Fantasy|Ro...,0.938,0.473,0.752
2210,2294,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy,1.000,0.304,0.722
6272,6377,Finding Nemo (2003),Adventure|Animation|Children|Comedy,0.869,0.460,0.706
3923,4016,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.000,0.260,0.704
2271,2355,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy,0.869,0.437,0.696
32293,134853,Inside Out (2015),Adventure|Animation|Children|Comedy|Drama|Fantasy,0.975,0.233,0.678
11899,53121,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy,1.000,0.184,0.674
5122,5218,Ice Age (2002),Adventure|Animation|Children|Comedy,0.869,0.340,0.658


In [10]:
hybrid_recommend(
    "The Dark Knight (2008)",
    10
)

"Movie 'The Dark Knight (2008)' not found."

In [12]:
hybrid_recommend(
    "Toy Story 2 (1999)",
    10
)

,movieId,title,genres,content_score,collaborative_score,hybrid_score
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1.000000,0.524543,0.809817
4791,4886,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,1.000000,0.502706,0.801082
4212,4306,Shrek (2001),Adventure|Animation|Children|Comedy|Fantasy|Ro...,0.937954,0.493366,0.760119
2271,2355,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy,0.869137,0.545829,0.739814
2210,2294,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy,1.000000,0.349222,0.739689
3923,4016,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.000000,0.329644,0.731858
6272,6377,Finding Nemo (2003),Adventure|Animation|Children|Comedy,0.869137,0.463729,0.706974
11899,53121,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy,1.000000,0.220024,0.688009
2902,2987,Who Framed Roger Rabbit? (1988),Adventure|Animation|Children|Comedy|Crime|Fant...,0.841957,0.447953,0.684356
5122,5218,Ice Age (2002),Adventure|Animation|Children|Comedy,0.869137,0.376614,0.672128


In [11]:
hybrid_config = {
    "content_weight": 0.6,
    "collaborative_weight": 0.4
}

joblib.dump(
    hybrid_config,
    "../models/hybrid_config.pkl"
)

print("Hybrid configuration saved successfully!")

Hybrid configuration saved successfully!
